# 4. Feature Engineering
Tiếp nối từ `02_cleaning.ipynb` — đọc lại checkpoint `cleaned.parquet` (nhánh riêng
từ `03_eda_after_cleaning.ipynb`, không phụ thuộc notebook đó).

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
%matplotlib inline

In [ ]:
df = pd.read_parquet("data/interim/cleaned.parquet")
print(f"Loaded checkpoint: {df.shape}")

In [ ]:
feature_cols = df.filter(regex="ind_.*ult.*").columns
feature_cols

## 6. Feature Engineering
Gộp toàn bộ các bước tạo feature mới vào một mạch: lag sản phẩm → lag/cờ thay đổi cho các cột hành vi → nhãn Added/Dropped/Maintained → long format → rút gọn lag dư thừa.

### 6.1 Lag features cho sản phẩm (lag 1–5)
Với mỗi sản phẩm, tạo 5 cột lag (trạng thái 1–5 tháng trước) để dùng làm feature và để tính Added/Dropped/Maintained ở bước sau.

In [ ]:
assert df[feature_cols].isin([0, 1]).all().all(), (
    "feature_cols không còn là 0/1 nhị phân — có thể đã chạy nhầm cell "
    "Added/Dropped trước cell lag này, hoặc kernel còn state cũ. "
    "Restart kernel và Run All lại từ đầu."
)

N_LAGS = 5
lag_dict = {}
for col in feature_cols:
    for lag in range(1, N_LAGS + 1):
        lag_dict[f"{col}_lag_{lag}"] = df.groupby("ncodpers")[col].shift(lag).fillna(0).astype(int)

lag_df = pd.DataFrame(lag_dict)
df = pd.concat([df, lag_df], axis=1)

### 6.2 Lag + cờ "changed" cho `segmento` / `ind_actividad_cliente` / `tiprel_1mes`

Các cột này phản ánh trạng thái/quan hệ của khách hàng theo tháng — bản thân sự **thay đổi** (vừa active lại, vừa đổi segment, đổi loại quan hệ...) là tín hiệu cross-sell mạnh hơn giá trị hiện tại (ví dụ khách vừa active lại có khả năng cao sắp phát sinh giao dịch/mua sản phẩm mới). Không cần đủ 5 lag như sản phẩm, chỉ cần `lag_1` + cờ nhị phân `changed` là đủ để bắt tín hiệu này.

In [ ]:
df = df.sort_values(["ncodpers", "fecha_dato"])

behavior_cols = [c for c in ["segmento", "ind_actividad_cliente", "tiprel_1mes"] if c in df.columns]
for col in behavior_cols:
    lag_col = f"{col}_lag_1"
    df[lag_col] = df.groupby("ncodpers")[col].shift(1)
    df[f"{col}_changed"] = (df[col] != df[lag_col]).astype(int)
    # Tháng đầu tiên của mỗi khách hàng không có lag -> không tính là "changed"
    df.loc[df[lag_col].isnull(), f"{col}_changed"] = 0

df[[f"{c}_changed" for c in behavior_cols]].mean()

### 6.2b — Encode categorical 

In [ ]:
# One-hot cho các cột nominal cardinality thấp
# Fix: thêm segmento_lag_1 / tiprel_1mes_lag_1 (bản tháng trước, tạo ở 6.2)
# vào cùng danh sách encode -- trước đó bị sót, vẫn còn ở dạng string/NaN thô
# nên không đưa thẳng vào model được.
low_card_encode = [
    "segmento", "sexo", "tiprel_1mes", "indrel_1mes",
    "segmento_lag_1", "tiprel_1mes_lag_1",
]
low_card_encode = [c for c in low_card_encode if c in df.columns]

# NaN ở *_lag_1 (khách chưa có tháng trước đó, vd. tháng đầu xuất hiện) ->
# coi là "UNKNOWN", nhất quán với cách xử lý missing của bản hiện tại
# (đã điền "UNKNOWN" ở bước 4.12).
for col in low_card_encode:
    if df[col].isnull().any():
        df[col] = df[col].astype(object).fillna("UNKNOWN")

cols_before_onehot = set(df.columns)
df = pd.get_dummies(df, columns=low_card_encode, prefix=low_card_encode, dtype="int8")
# Lấy đúng tên cột dummy mới sinh ra bằng cách diff trước/sau df.columns,
# KHÔNG dùng c.startswith(prefix) để gom cột: "segmento" và "tiprel_1mes"
# cũng là tiền tố của các cột lag/changed đã tạo ở bước 6.2
# (segmento_lag_1, tiprel_1mes_changed, ...) -> lọc theo prefix sẽ vô tình
# gom nhầm các cột đó vào nhóm one-hot.
onehot_cols = sorted(set(df.columns) - cols_before_onehot)

# Frequency encoding cho cột nominal cardinality cao
high_card_cols = ["canal_entrada", "pais_residencia", "nomprov"]
high_card_cols = [c for c in high_card_cols if c in df.columns]

freq_cols = []
for col in high_card_cols:
    freq = df[col].value_counts(normalize=True)
    df[f"{col}_freq"] = df[col].map(freq).astype("float32")
    freq_cols.append(f"{col}_freq")

# Bỏ cột string gốc sau khi đã có bản freq-encoded, tránh còn sót string
# cardinality cao trong output cuối (df_long / parquet) mà model không
# dùng trực tiếp được -- nếu muốn giữ lại để dễ debug/đọc, có thể bỏ dòng
# drop này, feature_columns ở cell dưới vẫn chỉ chọn đúng cột *_freq.
df = df.drop(columns=high_card_cols)

In [ ]:
# Cột binary S/N -> map thẳng 0/1
# Lưu ý: "indfall" KHÔNG còn nằm trong binary_cols nữa. Ở bước 4.1 (lọc
# khách hàng đã mất) toàn bộ dòng indfall == "S" đã bị loại khỏi df, nên
# tại thời điểm này cột indfall chỉ còn duy nhất giá trị "N" -- encode ra
# sẽ là một cột hằng số toàn 0, không còn giá trị dự đoán. Bỏ hẳn cột này
# thay vì giữ lại một feature "chết".
binary_cols = ["indresi", "indext"]
binary_cols = [c for c in binary_cols if c in df.columns]
for col in binary_cols:
    df[col] = df[col].map({"S": 1, "N": 0}).fillna(0).astype("int8")

if "indfall" in df.columns:
    df = df.drop(columns=["indfall"])

# ind_empleado: cardinality thấp (~5 giá trị: A/B/F/N/S) -> one-hot
cols_before_ind_empleado = set(df.columns)
if "ind_empleado" in df.columns:
    df = pd.get_dummies(df, columns=["ind_empleado"], prefix="ind_empleado", dtype="int8")
ind_empleado_cols = sorted(set(df.columns) - cols_before_ind_empleado)

# ult_fec_cli_1t: gần như toàn NaN/"UNKNOWN" -> bản thân giá trị ngày không
# có ý nghĩa dự đoán nhiều, chỉ cần biết CÓ hay KHÔNG có giá trị (khách từng
# đổi trạng thái primary customer) là đủ tín hiệu
if "ult_fec_cli_1t" in df.columns:
    df["has_ult_fec_cli_1t"] = (~df["ult_fec_cli_1t"].isin([np.nan, "UNKNOWN"])).astype("int8")
    df = df.drop(columns=["ult_fec_cli_1t"])

In [ ]:
# Tổng hợp feature_columns -- dùng chính các biến vừa tạo ở 2 cell trên
# (onehot_cols, freq_cols, ind_empleado_cols, binary_cols), KHÔNG còn tham
# chiếu tới train_df / lag1_cols / demo_cols / cat_encoded_cols "từ trên
# trời rơi xuống" như bản gốc -- đó chính là nguyên nhân NameError khi chạy
# (những biến này chưa từng được định nghĩa ở đâu trong notebook).

# lag_1 của 24 sản phẩm: tín hiệu portfolio hiện tại, đã tạo ở bước 6.1
lag1_cols = [f"{col}_lag_1" for col in feature_cols]

# Biến demographic / hành vi dạng số
demo_cols = [
    "age", "antiguedad", "renta", "ind_nuevo", "indrel",
    "ind_actividad_cliente", "month",
] + [f"{c}_changed" for c in behavior_cols]
demo_cols = [c for c in demo_cols if c in df.columns]

# Categorical đã encode ở 2 cell trên (one-hot + frequency)
cat_encoded_cols = onehot_cols + freq_cols + ind_empleado_cols

extra_cols = ["has_ult_fec_cli_1t"] + binary_cols
extra_cols = [c for c in extra_cols if c in df.columns]

feature_columns = lag1_cols + demo_cols + cat_encoded_cols + extra_cols
print(f"Tổng số feature: {len(feature_columns)}")
feature_columns

### 6.3 Added / Dropped / Maintained

Với mỗi khách hàng và mỗi sản phẩm, xác định trong tháng đó khách hàng đã thêm
mới, huỷ bỏ, hay giữ nguyên sản phẩm — bằng cách lấy diff giữa các tháng liên
tiếp. **Đảm bảo sort theo (`ncodpers`, `month_id`) trước khi diff** để tránh
trường hợp dữ liệu không theo đúng thứ tự thời gian (bản gốc giả định thứ tự đã
đúng sẵn).

In [ ]:
unique_months = df["fecha_dato"].drop_duplicates().sort_values().reset_index(drop=True)
month_id_map = {date: i + 1 for i, date in enumerate(unique_months)}
df["month_id"] = df["fecha_dato"].map(month_id_map)
df = df.sort_values(["ncodpers", "month_id"])

# Vectorized: dùng lag_1 đã tính ở bước lag feature, không cần groupby().transform() nữa
# Fix: tách "Maintained" thành 2 loại rõ ràng thay vì gộp chung, vì ý nghĩa khác hẳn nhau:
#   - Maintained_NotOwned (prev=0, current=0): chưa sở hữu, vẫn chưa sở hữu -> nhãn 0 quan trọng
#     nhất cho task "dự đoán sản phẩm MỚI", KHÔNG được lọc bỏ.
#   - Maintained_Owned (prev=1, current=1): đã sở hữu từ trước, vẫn giữ -> không thuộc phạm vi
#     task "thêm sản phẩm mới" (khách đã có sẵn rồi).
for col in feature_cols:
    lag_col = f"{col}_lag_1"
    current = df[col]
    prev = df[lag_col]  # đã fillna(0) sẵn từ bước tạo lag
    df[col] = np.select(
        [
            (current == 1) & (prev == 0),
            (current == 0) & (prev == 1),
            (current == 1) & (prev == 1),
        ],
        ["Added", "Dropped", "Maintained_Owned"],
        default="Maintained_NotOwned",
    )

### 6.4 Chuyển sang long format (melt)

**Fix (bug nghiêm trọng)**: bản gốc lọc bỏ hết các dòng "Maintained" ở bước này, chỉ giữ
lại Added/Dropped. Nhưng bài toán gốc là dự đoán sản phẩm **MỚI** sẽ được thêm — cần có
nhãn 0 (khách chưa sở hữu tháng trước, vẫn chưa sở hữu tháng này = `Maintained_NotOwned`)
thì model mới học được thế nào là "bình thường" trước khi phân biệt được trường hợp sắp
mua thêm. Lọc bỏ hết "Maintained" nghĩa là mất hết nhãn 0 đúng nghĩa, chỉ còn toàn các sự
kiện thay đổi (Added/Dropped) — model sẽ học sai bài toán.

**Cách dùng ở checkpoint 2** (train model "recommend sản phẩm mới"): lọc lấy các dòng có
`status` thuộc `{Added, Maintained_NotOwned}` (tức prev=0 — chưa sở hữu trước đó), rồi gán
`label = 1` nếu `Added`, `label = 0` nếu `Maintained_NotOwned`. Các dòng `Dropped` /
`Maintained_Owned` (đã sở hữu từ trước) nằm ngoài phạm vi task này, nhưng vẫn giữ lại ở
đây phòng khi cần cho phân tích churn.

Lưu ý: giữ toàn bộ sẽ làm file lớn hơn nhiều so với bản trước (chỉ ~21.8k dòng vì đã lọc
mất phần lớn dữ liệu) — với ~7.6k khách × ~14 tháng × 24 sản phẩm sẽ ra khoảng vài triệu
dòng. Nếu quá nặng cho máy, có thể downsample `Maintained_NotOwned` sau (giữ toàn bộ
Added/Dropped + sample ngẫu nhiên một phần Maintained_NotOwned), nhưng nên làm ở bước
train (checkpoint 2), không nên lọc mất tại bước lưu dữ liệu gốc này.

In [ ]:
df_long = df.melt(
    id_vars=[c for c in df.columns if c not in feature_cols],
    value_vars=list(feature_cols),
    var_name="product",
    value_name="status",
)

# Fix: KHÔNG lọc bỏ "Maintained_*" nữa — giữ toàn bộ để có đủ nhãn 0 cho training.
print(df_long["status"].value_counts())
df_long.shape

### 6.5 Rút gọn lag dư thừa

Sau melt, mỗi dòng chỉ nói về **1 sản phẩm** (cột `product`) nhưng vẫn mang theo `lag_2`–`lag_5` của cả 24 sản phẩm khác — dư thừa. Giữ nguyên `lag_1` của toàn bộ 24 sản phẩm (tín hiệu portfolio hiện tại, quan trọng cho cross-sell), chỉ rút gọn `lag_2`–`lag_5` về đúng 4 cột `self_lag_2`…`self_lag_5` ứng với sản phẩm đang xét ở từng dòng.

In [ ]:
product_list = list(feature_cols)
col_index = {p: i for i, p in enumerate(product_list)}
prod_idx = df_long["product"].map(col_index).to_numpy()
row_idx = np.arange(len(df_long))

for lag in range(2, N_LAGS + 1):
    lag_cols = [f"{p}_lag_{lag}" for p in product_list]
    lag_matrix = df_long[lag_cols].to_numpy()
    df_long[f"self_lag_{lag}"] = lag_matrix[row_idx, prod_idx]
    df_long = df_long.drop(columns=lag_cols)

print(f"Shape sau khi rút gọn lag: {df_long.shape}")
df_long.filter(regex="lag").columns.tolist()

## 7. Lưu kết quả

In [ ]:
LOCAL_PATH = "cleaned_long_format.parquet"
df_long.to_parquet(LOCAL_PATH, engine="pyarrow", index=False)

df_long.head()